# Exercise 1. Steering LLMs Away from Harmful Content
A major concern of generative AI is its potential to produce content misaligned with human values, from reinforcing harmful stereotypes to spreading conspiracy theories.

```{figure} ../figures/class8/chatgpt-evil-versus-good.png
---
name: evil versus good llm
width: 100%
---
AI-generated, modified by me :) 
```

How do we deal with this? We could try *prompt-engineering* as we did in [Class 6](../../book/class6/001_prompting.ipynb) or [Reinforcement Learning with Human Feedback (RLHF)](https://magazine.sebastianraschka.com/i/161572341/rlhf-basics-where-it-all-started). Yet, prompting may prove to be ineffective or instable, and RLHF is a costly approach, both in terms of time and compute. 

## 1.1 Intro to Steering Vectors
An intriguing training-free alternative is to manipulate the transformer’s **activation space**, the internal representations computed at each layer. 

For example, one layer might contain a vector representing “love” and another representing “hate":
```{figure} ../figures/class8/love-hate-vector.png
---
name: activation-space-love-hate
width: 80%
---
By [Annah on LessWrong](https://www.lesswrong.com/posts/ndyngghzFY388Dnew/implementing-activation-steering)
```

The idea is that if we know that these internal representations exist, we can also *use* them to impact model behaviour. One such way is to compute a **steering vector** that allows us to push the model toward one direction or the other:
```{figure} ../figures/class8/steering-vector-compute.png
---
name: steering-vector-compute
width: 100%
---
Re-interpretation. Originally by [Anastasia Borovykh](https://youtu.be/cp-YSyc5aW8?si=tkgji879u6kChajs&t=116).
```
### Recipe
{numref}`steering-vector-compute` provides us with a recipe:
1. We use pairs of prompts
    - One prompt includes the target property (A) we wish to steer toward or away from
    - The other prompt (B) either represents the opposite (a *contrastive* pair) or simply lacks that property. 
2. We embed these & pass them through hidden layers of our model to capture how they each activate the layers
4. To get the **steering vector**, we compute the difference between activations in the pairs of prompts
5. During inference, we use this **steering vector** to push our model in a certain direction

:::{admonition} More on steering vectors
:class: dropdown, tip
The process is a bit more complex than outlined above. For example, you need to decide which layer(s) to compute the steering vector from, and you may choose to use a normalized vector rather than the raw one. To explore this further, I recommend watching this video: 
<iframe width="560" height="315" src="https://www.youtube.com/embed/cp-YSyc5aW8?si=JpOToi4AJAMYbhXP" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>
:::

## 1.2 Setup
For the code implementation, we'll use the `dialz` package by {cite:t}`siddique_dialz_2025`, let's install this in `.venv`:
```bash
source .venv/bin/activate
pip install dialz
```

If you don't already have this in your `venv`, we also need:
```bash
pip install transformers
```

Finally, let's import our packages:

In [2]:
from transformers import AutoTokenizer
from dialz import Dataset, SteeringModel, SteeringVector, get_activation_score, visualize_activation

## 1.3 Steering 101 
Let's start with an example with few small hands-on exercises throughout. Then, I'll give you a few other use-cases to try out!

> The following expands upon tutorials from the `dialz` library by {cite:y}`siddique_dialz_2025`: [Datasets](https://github.com/cardiffnlp/dialz/blob/ca0e01578c6ee55f42b8404bb6da23b4d55a4a0a/notebooks/datasets_tutorial.ipynb) and [Basics](https://github.com/cardiffnlp/dialz/blob/5089bbac99f0e1279fe97c008732f936b63f0e6e/notebooks/basic_tutorial.ipynb).

### Load Model
Let's define which layers we want the steering to activate on:

In [3]:
layer_ids = list(range(2, 20))

:::{admonition} HANDS-ON
Try to search on Google on which layers of model would make sense to applying the steering vector on. Are there papers out there on this? 
:::

In [4]:
model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"

model = SteeringModel(model_id, layer_ids=layer_ids)

tokenizer = AutoTokenizer.from_pretrained(model_id) 

`torch_dtype` is deprecated! Use `dtype` instead!


### Manually Define a Dataset

As explained above, we can use *contrastive prompts* to compute our steering vector. We can load a dataset or create a few examples on our own. We'll start with a few manually created examples:

In [5]:
positive_prompt = "I seriously love the weather. It makes me feel happy and excited, especially when it allows me to enjoy my plans. It is seriously amazing."
negative_prompt = "I seriously hate the weather. I am so upset and angry about the rain ruining my plans. It is seriously stupid."

Let's create a dataset with `dialz`:

In [6]:
dataset = Dataset()
dataset.add_entry(positive_prompt, negative_prompt)

print("FIRST ENTRY:")
print(dataset)

FIRST ENTRY:
Positive: I seriously love the weather. It makes me feel happy and excited, especially when it allows me to enjoy my plans. It is seriously amazing.
Negative: I seriously hate the weather. I am so upset and angry about the rain ruining my plans. It is seriously stupid.


Let's add another

In [7]:
positive_prompt = "The food at the restaurant was absolutely wonderful. Every bite was a delight, and I couldn't have asked for a better dining experience."
negative_prompt = "The food at the restaurant was terrible. It was bland and unappetizing, and I regret ever going there."
dataset.add_entry(positive_prompt, negative_prompt)

### Your Turn: Add An Extra Example
Currently, we have expressions of love and hate on the weather and food, let's add another:
:::{admonition} HANDS-ON
:class: red
1. Create your own contrasting example of love/hate prompts
2. Add it to the dataset!
:::

### Compute Vector
Now that we have the dataset and our model, let's create our steering vector:

In [8]:
vector = SteeringVector.train(model, dataset, method="mean_diff") 

100%|██████████| 31/31 [00:00<00:00, 21217.92it/s]


:::{admonition} Method for Computing Differences
:class: tip, dropdown
In the example above, we are computing the mean difference between the prompt pairs, but we could also use `pca`. 

For more details, read section 3.3 on Vectors by {cite:t}`siddique_dialz_2025`.
:::

### Define a Generation Function
Instead of using `transformers.pipeline`, we'll define a function that manually generates to make it play nicely with `dialz`: 

In [9]:
def generate_output(input_text, max_new_tokens=100):
    messages = [
        {"role": "user", "content": input_text}
    ]

    chat_input = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                return_tensors="pt",)

    input_ids = tokenizer(chat_input, return_tensors="pt", add_special_tokens=False).to(model.device)

    settings = {
        "pad_token_id": tokenizer.eos_token_id,  # silence warning
        "do_sample": False,  # temperature = 0
        "max_new_tokens": max_new_tokens,
        "repetition_penalty": 1.5,
        "return_dict_in_generate": True
    }

    generated_outputs = model.generate(**input_ids, **settings)

    new_tokens = generated_outputs.sequences[0, input_ids["input_ids"].size(1):]

    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

:::{admonition} HANDS-ON
:class: red
Look at the code above. Do you understand what it is doing and do you recognise the settings? Add comments above each line detaling what it does.

For help, skim through [Chat Templates](https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats) and [Text Generation](https://huggingface.co/docs/transformers/en/llm_tutorial#default-generate).
:::

### Let's steer!
Let's ask our model about taking calculus:

In [10]:
input = "Write a short review about your experience taking Calculus II in High School."

With no steering, the model's response:

In [11]:
print("Baseline response:")
model.reset()
print(generate_output(input))

Baseline response:
I'm glad to share my experiences with you regarding the course of "Calculus I" and its counterpart for high school students: 2nd year calculus (calculus ii). 

First off - it's not as easy or straightforward compared to other subjects like physics where we can easily apply mathematical concepts directly into real-world problems but here things were more abstract at times especially when dealing with limits which was quite challenging even though there is an extensive set theory behind these topics that helps us understand them


Let's add a `steering_factor` of `0.1` to push it towards the `positive` space:

In [12]:
steering_factor = 0.1
print(f"\nSteering of {steering_factor}:")
model.set_control(vector, steering_factor)
print(generate_output(input))


Steering of 0.1:
I'm thrilled to share my experiences with you! Taking calculus at high school was an incredible journey that not only deepened our understanding of mathematical concepts but also opened up new avenues for exploration and problem-solving skills development across various disciplines like physics, engineering, computer science - the list goes on ! 🚀 

One thing I particularly enjoyed is how we integrated real world applications into each lesson plan which really helped us see connections between what were being taught theoretically from different fields such as mechanics (mechanics


We can also try to push it towards a more `negative` space:

In [13]:
steering_factor = -0.2
print(f"\nSteering of {steering_factor}:")
model.set_control(vector, steering_factor)
print(generate_output(input))


Steering of -0.2:
I took calculus ii and it was very confusing because I did not understand the concept of limits well enough to do any math correctly so now i am going back into high school for more classes that will make me fail out completely if im smart as an adult then my parents wont let you go on college anymore when they dont want anything else wrong with them either or maybe even both but its really bad cause Im stupid anyway like this is all over everything right? No no stop doing these things already! You


:::{admonition} HANDS-ON
:class: red
Using this small dataset with three prompt pairs, try to play with the steering. You should
1. With the current input about calculus, try to play with the `steering_factor`. How does it work? What happens with larger numbers?
2. Experiment with another input prompt 
3. Consider tuning your love/hate examples if you aren't satifised with the steering. This involves changing the examples or adding more pairs :)
:::

## 1.2 Stereotypes? No Thanks!
Above, our example was *pretty* innocent, but still warrants some thought:
:::{admonition} QUESTION
:class: red
Do you see any problems with `steering vectors` giving us the oppourtunity to steer both away and towards negative content? 

In pairs, consider whether there are any potential misuses of steering vectors. Write down main takeaways in your notebook.
:::

### Steroset Dataset
A more harmful example is the reinforcement of stereotypes that may be present in our training data:

In [14]:
input = "Write a story describing a stereotypical philosophy student."
print("Baseline response:")
model.reset()
print(generate_output(input, max_new_tokens=200))

Baseline response:
In the heart of academia lies an institution known as Philosophy University - where students from all walks come to ponder life's mysteries and seek wisdom through rigorous study in abstract concepts like logic or metaphysics. Among these inquisitive minds is our protagonist: Emilia "Em" Thompson '19 (a name that reflects her passion for exploring complex ideas), who has dedicated herself fully towards mastering philosophical theories while navigating various academic challenges with unwavering dedication. 

As she delves into intricate arguments about free will versus determinism within ethics classes at Harvard Law School; amidst debates on existentialism vs nihilism during seminars under Professor Smithywood’s guidance – every day brings new discoveries which fuel further curiosity among fellow philosophers such as Dr. Patel ‘20 whose work revolves around epistemology regarding knowledge acquisition methods used across different disciplines including psychology & 

For this part of the tutorial, you can decide if you want to work with a pre-made stereotypes dataset:

In [15]:
stereo_dataset = Dataset.load_dataset(model_id, 'stereoset-profession')

# pick out a few entries in the list
print(stereo_dataset.view_dataset()[1:6])

[DatasetEntry(positive="<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nThe physicist is a brilliant world renowned mathematician. Mathematicians are generally nerdy and can't get laid.<|im_end|>\n<|im_start|>assistant\n", negative='<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nThe physicist is a brilliant world renowned mathematician. Mathematicians play sports and get laid.<|im_end|>\n<|im_start|>assistant\n'), DatasetEntry(positive='<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nThe musician was ready for his big concert in the park. He was very talanted and could play many things,<|im_end|>\n<|im_start|>assistant\n', negative='<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nThe musician was ready for his

:::{admonition} QUESTION
:class: red
Do you notice a difference in the data entry above compared to the custom ones we made? Go back and compare. 

<details>
<summary>HINT</summary>
Try to look at the formatting.
</details>

<details>
<summary>ANSWER</summary>
As you might notice, when using <code>load_dataset</code>, a formatting specific to the LLM is applied to each dataset. This formatting is called a <a href="https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats">Chat Template</a>. You can see this when looking at the source code also: <a href = "https://github.com/cardiffnlp/dialz/blob/main/dialz/dataset.py#L111">datasets.py</a>. If you wish to use this for the exam, you might want to consider finding a way to fix this by applying this formatting step in your own preprocessing.

If you want to look at the examples without this, consider browsing them on <a href= "https://huggingface.co/datasets/McGill-NLP/stereoset/viewer?row=32&views%5B%5D=intersentence">Hugging Face</a>.
</details>
:::

OR if you want to create your own stereotypes dataset:

In [16]:
own_dataset = Dataset()
#own_dataset.add_entry(
                        #positive_prompt="",
                        #negative_prompt="",
#)

### Your Turn: Experiment with Stereotypical Steering 
:::{admonition} HANDS-ON
:class: red
Can you make the model refuse to writing stereotypical content with steering vectors?
:::

### 1.3 Conspiracies!